In [17]:
from jqdata import *
import datetime
import pandas as pd 
stock_df = get_all_securities(types=['stock'], date=None)
stock_df['code'] = stock_df.index
# 过滤科创和创业板，这个看个人喜欢
stock_df = stock_df[~(stock_df['code'].str.startswith("3") | stock_df['code'].str.startswith("68"))]

stock_lists = list(stock_df['code'])
stock_df.head()

# 获取交易日列表，因为财务数据限制了单次获取条数，干脆分天获取了
start_date = '2022-01-01'

# 自己设置，简单示例
end_date = '2022-03-01'
trade_days = get_trade_days(start_date=start_date, end_date=end_date, count=None)
trade_days[:5],trade_days[-5:]

(array([2022-01-04, 2022-01-05, 2022-01-06, 2022-01-07, 2022-01-10],
       dtype=object),
 array([2022-02-23, 2022-02-24, 2022-02-25, 2022-02-28, 2022-03-01],
       dtype=object))

In [18]:
# 量价信息
price_df = get_price(stock_lists, start_date=start_date, end_date=end_date, frequency='daily', fields=['open','close','low','high','volume','money',
        'high_limit','low_limit','pre_close','paused'], skip_paused=False, fq='post', count=None,round=True,panel=False)
price_df.head()


,time,code,open,close,low,high,volume,money,high_limit,low_limit,pre_close,paused
0,2022-01-04,000001.XSHE,2005.93,2027.84,1969.42,2027.84,960621.0,1.918887e+09,2206.77,1805.09,2005.93,0.0
1,2022-01-05,000001.XSHE,2018.10,2087.48,2014.45,2096.00,1611906.0,3.344125e+09,2231.11,1824.57,2027.84,0.0
2,2022-01-06,000001.XSHE,2082.61,2083.83,2069.23,2102.09,910198.0,1.896536e+09,2296.84,1879.34,2087.48,0.0
3,2022-01-07,000001.XSHE,2081.40,2093.57,2076.53,2103.31,925599.0,1.937711e+09,2291.97,1875.69,2083.83,0.0
4,2022-01-10,000001.XSHE,2104.52,2092.35,2072.88,2120.35,747437.0,1.563415e+09,2302.93,1884.21,2093.57,0.0


In [19]:
has_limit_stocks = list(price_df.query("close == high_limit")['code'].unique())

In [20]:
stock_lists = has_limit_stocks

In [21]:
st_df = get_extras('is_st', stock_lists, start_date=start_date, end_date=end_date)
st_df = st_df.stack().reset_index().rename(columns={'level_0':'time','level_1':'code',0:'is_st'})
st_df.head()

,time,code,is_st
0,2022-01-04,000004.XSHE,False
1,2022-01-04,000008.XSHE,False
2,2022-01-04,000014.XSHE,False
3,2022-01-04,000017.XSHE,False
4,2022-01-04,000020.XSHE,False


# 数据合并

In [22]:
merge_df = price_df.merge(stock_df,how='left',on=['code'],suffixes=['','_x'])
merge_df = merge_df.merge(st_df,how='left',on=['code','time'],suffixes=['','_x'])
merge_df['list_days'] = (merge_df['time'] - pd.to_datetime(merge_df['start_date'])).dt.days
merge_df.head()

,time,code,open,close,low,high,volume,money,high_limit,low_limit,pre_close,paused,display_name,name,start_date,end_date,type,is_st,list_days
0,2022-01-04,000001.XSHE,2005.93,2027.84,1969.42,2027.84,960621.0,1.918887e+09,2206.77,1805.09,2005.93,0.0,平安银行,PAYH,1991-04-03,2200-01-01,stock,NaN,11234
1,2022-01-05,000001.XSHE,2018.10,2087.48,2014.45,2096.00,1611906.0,3.344125e+09,2231.11,1824.57,2027.84,0.0,平安银行,PAYH,1991-04-03,2200-01-01,stock,NaN,11235
2,2022-01-06,000001.XSHE,2082.61,2083.83,2069.23,2102.09,910198.0,1.896536e+09,2296.84,1879.34,2087.48,0.0,平安银行,PAYH,1991-04-03,2200-01-01,stock,NaN,11236
3,2022-01-07,000001.XSHE,2081.40,2093.57,2076.53,2103.31,925599.0,1.937711e+09,2291.97,1875.69,2083.83,0.0,平安银行,PAYH,1991-04-03,2200-01-01,stock,NaN,11237
4,2022-01-10,000001.XSHE,2104.52,2092.35,2072.88,2120.35,747437.0,1.563415e+09,2302.93,1884.21,2093.57,0.0,平安银行,PAYH,1991-04-03,2200-01-01,stock,NaN,11240


In [23]:
def cal_lb_count(xs):
    """
    计算股票连板数
    """
    x = 0
    for d in xs[::-1]:
        if d == 0:
            return x
        else:
            x += 1
            
    return x 
    


In [24]:
merge_df['open_valid1'] = merge_df.groupby('code')['open'].shift(-1)
merge_df['is_close_high_limit'] = (merge_df['close'] == merge_df['high_limit']) * (1-merge_df['paused'])
merge_df['is_low_high_limit'] = (merge_df['low'] == merge_df['high_limit']) * (1-merge_df['paused'])
merge_df['is_low_low_limit'] = (merge_df['low'] == merge_df['low_limit']) * (1-merge_df['paused'])
merge_df['is_close_low_limit'] = (merge_df['close'] == merge_df['low_limit']) * (1-merge_df['paused'])
merge_df['最大连板数'] = merge_df.groupby('code').rolling(20)['is_close_high_limit'].apply(lambda x:cal_lb_count(x)).values
merge_df['open_pct_valid_1'] = merge_df['open_valid1']/merge_df['close']
merge_df.head()

/opt/conda/lib/python3.6/site-packages/ipykernel_launcher.py:6: FutureWarning: Currently, 'apply' passes the values as ndarrays to the applied function. In the future, this will change to passing it as Series objects. You need to specify 'raw=True' to keep the current behaviour, and you can pass 'raw=False' to silence this warning
  


,time,code,open,close,low,high,volume,money,high_limit,low_limit,pre_close,paused,display_name,name,start_date,end_date,type,is_st,list_days,open_valid1,is_close_high_limit,is_low_high_limit,is_low_low_limit,is_close_low_limit,最大连板数,open_pct_valid_1
0,2022-01-04,000001.XSHE,2005.93,2027.84,1969.42,2027.84,960621.0,1.918887e+09,2206.77,1805.09,2005.93,0.0,平安银行,PAYH,1991-04-03,2200-01-01,stock,NaN,11234,2018.10,0.0,0.0,0.0,0.0,NaN,0.995197
1,2022-01-05,000001.XSHE,2018.10,2087.48,2014.45,2096.00,1611906.0,3.344125e+09,2231.11,1824.57,2027.84,0.0,平安银行,PAYH,1991-04-03,2200-01-01,stock,NaN,11235,2082.61,0.0,0.0,0.0,0.0,NaN,0.997667
2,2022-01-06,000001.XSHE,2082.61,2083.83,2069.23,2102.09,910198.0,1.896536e+09,2296.84,1879.34,2087.48,0.0,平安银行,PAYH,1991-04-03,2200-01-01,stock,NaN,11236,2081.40,0.0,0.0,0.0,0.0,NaN,0.998834
3,2022-01-07,000001.XSHE,2081.40,2093.57,2076.53,2103.31,925599.0,1.937711e+09,2291.97,1875.69,2083.83,0.0,平安银行,PAYH,1991-04-03,2200-01-01,stock,NaN,11237,2104.52,0.0,0.0,0.0,0.0,NaN,1.005230
4,2022-01-10,000001.XSHE,2104.52,2092.35,2072.88,2120.35,747437.0,1.563415e+09,2302.93,1884.21,2093.57,0.0,平安银行,PAYH,1991-04-03,2200-01-01,stock,NaN,11240,2100.87,0.0,0.0,0.0,0.0,NaN,1.004072


## 衍生特征

In [25]:
# 概念龙头特征计算
result_dfs = []
from tqdm import tqdm
for t,tmp_df in tqdm(merge_df.query("is_close_high_limit == 1").groupby('time')):
    security = list(tmp_df['code'])
    date = t
    cs = get_concept(security, date=date)
    dfs = []
    for code,concept_data in cs.items():
        for c in concept_data['jq_concept']:
            if c['concept_name'] not in ['转融券标的', '融资融券', '深股通', '沪股通']:
                dfs.append({'code':code,'concept':c['concept_name']})
    concept_df = pd.DataFrame(dfs)

    a = concept_df['concept'].value_counts()
    hot_concept_limit_count = a.max()
    hot_concepts = list(a[a==hot_concept_limit_count].index)
    hot_concept_stocks = list(concept_df[concept_df['concept'].isin(hot_concepts)].code)
    hot_concept_limit_count,hot_concepts,hot_concept_stocks
    result_dfs.append({"time":t,
                        "概念龙头涨停数":hot_concept_limit_count,
                      "概念龙头列表":hot_concepts,
                       "概念龙头股票列表":hot_concept_stocks,
                      })


hot_concept_df = pd.DataFrame(result_dfs)
hot_concept_df.head()

100%|██████████| 36/36 [00:52<00:00,  1.35s/it]


,time,概念龙头列表,概念龙头涨停数,概念龙头股票列表
0,2022-01-04,[创投],15,"[000558.XSHE, 000863.XSHE, 000955.XSHE, 002083..."
1,2022-01-05,"[互联网+, 文化传媒, 创投]",7,"[000530.XSHE, 000558.XSHE, 000665.XSHE, 000665..."
2,2022-01-06,[创投],11,"[000558.XSHE, 000607.XSHE, 000812.XSHE, 002101..."
3,2022-01-07,[新股与次新股],7,"[001219.XSHE, 001296.XSHE, 003037.XSHE, 603176..."
4,2022-01-10,[文化传媒],9,"[000665.XSHE, 000673.XSHE, 000863.XSHE, 002188..."


In [26]:
# 当天市场特征
day_feature_df = pd.DataFrame(merge_df.groupby('time')['最大连板数'].max().rename("市场最大连板数"))
day_feature_df['一字板数量'] = merge_df.groupby('time')['is_low_high_limit'].sum()
day_feature_df['涨停板数量'] = merge_df.groupby('time')['is_close_high_limit'].sum()
day_feature_df['涨停溢价'] = merge_df.query("is_close_high_limit == 1").groupby('time')['open_pct_valid_1'].mean()
day_feature_df['近3天市场最大连板数'] = day_feature_df['市场最大连板数'].rolling(3).max()
day_feature_df['近3天市场平均涨停溢价'] = day_feature_df['涨停溢价'].rolling(3).mean()
day_feature_df['近3天市场最高涨停溢价'] = day_feature_df['涨停溢价'].rolling(3).max()
day_feature_df['近3天市场涨停溢价std'] = day_feature_df['涨停溢价'].rolling(3).std()
day_feature_df['近3天市场最低涨停溢价'] = day_feature_df['涨停溢价'].rolling(3).min()
day_feature_df['今天是否等于近3天市场最大连板数'] = day_feature_df['市场最大连板数']==day_feature_df['近3天市场最大连板数']
day_feature_df.tail(10)

,市场最大连板数,一字板数量,涨停板数量,涨停溢价,近3天市场最大连板数,近3天市场平均涨停溢价,近3天市场最高涨停溢价,近3天市场涨停溢价std,近3天市场最低涨停溢价,今天是否等于近3天市场最大连板数
time,,,,,,,,,,
2022-02-16,8.0,7.0,73.0,1.024114,8.0,1.020295,1.024114,0.005328,1.014209,True
2022-02-17,4.0,7.0,56.0,1.012625,8.0,1.019767,1.024114,0.006234,1.012625,False
2022-02-18,5.0,12.0,66.0,1.034559,8.0,1.023766,1.034559,0.010971,1.012625,False
2022-02-21,6.0,17.0,101.0,1.019365,6.0,1.022183,1.034559,0.011235,1.012625,True
2022-02-22,7.0,16.0,58.0,1.029655,7.0,1.027860,1.034559,0.007754,1.019365,True
2022-02-23,7.0,13.0,94.0,1.019945,7.0,1.022988,1.029655,0.005781,1.019365,True
2022-02-24,8.0,6.0,44.0,1.010147,8.0,1.019916,1.029655,0.009754,1.010147,True
2022-02-25,9.0,9.0,70.0,1.023103,9.0,1.017732,1.023103,0.006755,1.010147,True
2022-02-28,10.0,8.0,55.0,1.022850,10.0,1.018700,1.023103,0.007408,1.010147,True


In [27]:
day_feature_df = day_feature_df.merge(hot_concept_df,how='left',left_index=True,right_on=['time'])
day_feature_df.head()

,市场最大连板数,一字板数量,涨停板数量,涨停溢价,近3天市场最大连板数,近3天市场平均涨停溢价,近3天市场最高涨停溢价,近3天市场涨停溢价std,近3天市场最低涨停溢价,今天是否等于近3天市场最大连板数,time,概念龙头列表,概念龙头涨停数,概念龙头股票列表
0,NaN,12.0,120.0,1.031930,NaN,NaN,NaN,NaN,NaN,False,2022-01-04,[创投],15,"[000558.XSHE, 000863.XSHE, 000955.XSHE, 002083..."
1,NaN,12.0,66.0,1.019762,NaN,NaN,NaN,NaN,NaN,False,2022-01-05,"[互联网+, 文化传媒, 创投]",7,"[000530.XSHE, 000558.XSHE, 000665.XSHE, 000665..."
2,NaN,6.0,92.0,1.024754,NaN,1.025482,1.031930,0.006117,1.019762,False,2022-01-06,[创投],11,"[000558.XSHE, 000607.XSHE, 000812.XSHE, 002101..."
3,NaN,8.0,48.0,1.011404,NaN,1.018640,1.024754,0.006745,1.011404,False,2022-01-07,[新股与次新股],7,"[001219.XSHE, 001296.XSHE, 003037.XSHE, 603176..."
4,NaN,5.0,83.0,1.021142,NaN,1.019100,1.024754,0.006905,1.011404,False,2022-01-10,[文化传媒],9,"[000665.XSHE, 000673.XSHE, 000863.XSHE, 002188..."


In [28]:
# 只看今天有涨停的票
merge_df = merge_df.query("is_close_high_limit == 1").merge(day_feature_df,how='left',on=['time'],suffixes=['','_市场'])
merge_df.head()

,time,code,open,close,low,high,volume,money,high_limit,low_limit,pre_close,paused,display_name,name,start_date,end_date,type,is_st,list_days,open_valid1,is_close_high_limit,is_low_high_limit,is_low_low_limit,is_close_low_limit,最大连板数,open_pct_valid_1,市场最大连板数,一字板数量,涨停板数量,涨停溢价,近3天市场最大连板数,近3天市场平均涨停溢价,近3天市场最高涨停溢价,近3天市场涨停溢价std,近3天市场最低涨停溢价,今天是否等于近3天市场最大连板数,概念龙头列表,概念龙头涨停数,概念龙头股票列表
0,2022-01-05,000004.XSHE,149.22,163.74,149.22,163.74,1016150.0,1.624793e+08,163.74,133.95,148.85,0.0,国华网安,GHWA,1990-12-01,2200-01-01,stock,False,11358,164.56,1.0,0.0,0.0,0.0,NaN,1.005008,NaN,12.0,66.0,1.019762,NaN,NaN,NaN,NaN,NaN,False,"[互联网+, 文化传媒, 创投]",7,"[000530.XSHE, 000558.XSHE, 000665.XSHE, 000665..."
1,2022-01-26,000004.XSHE,165.75,165.75,165.75,165.75,752436.0,1.247148e+08,165.75,135.67,150.71,0.0,国华网安,GHWA,1990-12-01,2200-01-01,stock,False,11379,171.26,1.0,1.0,0.0,0.0,NaN,1.033243,NaN,2.0,59.0,1.009876,NaN,1.012040,1.014783,0.002504,1.009876,False,[华为],8,"[000004.XSHE, 002101.XSHE, 002104.XSHE, 002197..."
2,2022-02-18,000008.XSHE,69.63,77.42,68.82,77.42,3691277.0,2.722971e+08,77.42,63.44,70.43,0.0,神州高铁,SZGT,1992-05-07,2200-01-01,stock,False,10879,79.03,1.0,0.0,0.0,0.0,1.0,1.020796,5.0,12.0,66.0,1.034559,8.0,1.023766,1.034559,0.010971,1.012625,False,[数据中心],11,"[000815.XSHE, 002178.XSHE, 002316.XSHE, 002771..."
3,2022-02-15,000014.XSHE,77.46,84.61,76.09,84.61,2098460.0,1.727380e+08,84.61,69.22,76.91,0.0,沙河股份,SHGF,1992-06-02,2200-01-01,stock,False,10850,85.70,1.0,0.0,0.0,0.0,1.0,1.012883,7.0,7.0,65.0,1.022562,8.0,1.015570,1.022562,0.006421,1.009939,False,[文化传媒],8,"[000607.XSHE, 000793.XSHE, 002188.XSHE, 002542..."
4,2022-02-24,000017.XSHE,11.52,12.76,11.45,12.76,6797392.0,8.525072e+07,12.76,10.41,11.58,0.0,深中华A,SZHA,1992-03-31,2200-01-01,stock,False,10922,12.69,1.0,0.0,0.0,0.0,1.0,0.994514,8.0,6.0,44.0,1.010147,8.0,1.019916,1.029655,0.009754,1.010147,True,"[新股与次新股, 黄金]",6,"[000017.XSHE, 000587.XSHE, 001313.XSHE, 002155..."


In [29]:

# - 股票是否在概念龙头中
merge_df['股票在概念龙头'] = merge_df.apply(lambda row: row['code'] in row['概念龙头股票列表'],axis=1)